# U08 练习题

完成以下练习，过关标准：练习 8.3 数字反转准确率 >95%。

---

In [2]:
import torch
import torch.nn as nn
import random

PAD, SOS, EOS = 0, 1, 2
VOCAB_SIZE = 13
EMBED_DIM  = 16
HIDDEN_DIM = 64
BATCH_SIZE = 64
SEQ_LEN    = 5

## 练习 8.1：观察 Context Vector 的形状

创建一个 Encoder，运行前向传播，观察 `output` 和 `hidden` 的 shape。

**回答问题**：
1. `output` 和 `hidden` 的 shape 分别是什么？
2. 为什么 Encoder 只返回 `hidden` 而不返回 `output`？
3. `hidden` 的第一个维度（num_layers=1）表示什么？

In [3]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        return output, hidden   # 这里同时返回两个，用于观察


enc = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
src = torch.randint(3, 13, (BATCH_SIZE, SEQ_LEN))

output, hidden = enc(src)

# TODO: 打印 output 和 hidden 的 shape，并回答上面三个问题
print('output shape:', output.shape)
print('hidden shape:', hidden.shape)

# 你的回答（注释形式）：
# Q1：output.shape = (batch, seqlen, hidden) hidden.shape = (1, batch, hidden)
# Q2：因为encoder的返回值直接作为decoder的输入hn，output没有用到
# Q3：num_layers * bidirectional

output shape: torch.Size([64, 5, 64])
hidden shape: torch.Size([1, 64, 64])


## 练习 8.2：手写 Decoder 单步推理

补全 `DecoderStep` 类，实现单步解码。

要求：
- `forward(x, hidden)` 中，`x` 是 shape `(batch, 1)` 的 token 索引
- 返回 `logits (batch, vocab_size)` 和新的 `hidden`
- 不能直接复制 lesson 的代码，理解后手写

In [5]:
class DecoderStep(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # TODO 1: 定义三个层：Embedding, GRU, Linear
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, vocab_size)
        # 注意：GRU 每次处理一个 token，batch_first=True

    def forward(self, x, hidden):
        # x:      (batch, 1)
        # hidden: (1, batch, hidden_dim)
        # TODO 2: 三步走：Embedding -> GRU -> Linear
        x = self.embedding(x)
        output, hidden = self.gru(x)
        output = output.squeeze(1)
        output = self.linear(output)
        return output, hidden
        # 注意 GRU 输出 output shape 是 (batch, 1, hidden_dim)，需要 squeeze(1) 再过 Linear


# 测试
dec = DecoderStep(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
ctx = torch.zeros(1, BATCH_SIZE, HIDDEN_DIM)            # 模拟 context vector
tok = torch.full((BATCH_SIZE, 1), SOS, dtype=torch.long)

logits, new_hidden = dec(tok, ctx)
print('logits shape:     ', logits.shape)      # 期望: (64, 13)
print('new_hidden shape: ', new_hidden.shape)  # 期望: (1, 64, 64)

logits shape:      torch.Size([64, 13])
new_hidden shape:  torch.Size([1, 64, 64])


## 练习 8.3：完整 Seq2Seq 数字反转任务

补全 Seq2Seq 模型和训练循环，完成数字反转任务。

**过关标准**：val_acc > 95%

In [3]:
# ===== 数据生成（已给出）=====
def make_batch(batch_size, seq_len=SEQ_LEN):
    src = torch.randint(3, 13, (batch_size, seq_len))
    tgt = torch.zeros(batch_size, seq_len + 2, dtype=torch.long)
    tgt[:, 0] = SOS
    for i in range(seq_len):
        tgt[:, i + 1] = src[:, seq_len - 1 - i]
    tgt[:, seq_len + 1] = EOS
    return src, tgt # 原始数组，反转数组


# ===== Encoder（已给出）=====
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        return hidden


# ===== Decoder（已给出）=====
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc(output.squeeze(1))
        return logits, hidden


# ===== TODO: 补全 Seq2Seq =====
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len    = tgt.size(1)
        vocab_size = self.decoder.fc.out_features

        # TODO 1: 用 Encoder 得到 hidden（context vector）
        # encoder = Encoder(vocab_size, EMBED_DIM, HIDDEN_DIM)
        hidden = self.encoder(src)

        # TODO 2: 初始化 outputs 存储张量，shape=(batch, tgt_len, vocab_size) 
        outputs = torch.zeros(batch_size, tgt_len, vocab_size)

        # TODO 3: 初始化 input_token 为 tgt 的第一列（SOS）
        input_token = tgt[:,0]

        # TODO 4: 循环 t=1..tgt_len-1，每步调用 decoder
        #         - 把 logits 存入 outputs[:,t,:]
        #         - 用 teacher_forcing_ratio 决定下一步的 input_token
        for t in range(1, tgt_len):
            logits,hidden = self.decoder(input_token, hidden)
            outputs[:,t,:] = logits
            if random.random() < teacher_forcing_ratio:
                input_token = tgt[:,t]
            else:
                input_token = logits.argmax(1)

        return outputs

In [ ]:
# ===== TODO: 推理函数 =====
def predict(model, src):
    model.eval()
    with torch.no_grad():
        # TODO: 用 encoder 得到 hidden，然后逐步解码
        hidden = model.encoder(src)
        # 每步取 argmax 作为下一步输入
        outputs = []
        input_token = torch.full((src.size(0),), SOS, dtype=torch.long)
        for i in range(SEQ_LEN + 1):            
            logits, hidden = model.decoder(input_token,hidden)
            outputs.append(logits)
            input_token = logits.argmax(dim=1)
        # 返回 preds: (batch, SEQ_LEN+2) 的预测序列
        return torch.stack(outputs, dim=1).argmax(dim=-1)


# ===== 准确率（已给出）=====
def accuracy(model, n_batches=50):
    correct, total = 0, 0
    for _ in range(n_batches):
        src, tgt = make_batch(BATCH_SIZE)
        preds = predict(model, src)
        pred_seq = preds[:, :SEQ_LEN]
        true_seq = tgt[:, 1:SEQ_LEN+1]
        correct += (pred_seq == true_seq).all(dim=1).sum().item()
        total   += src.size(0)
    return correct / total


# ===== TODO: 训练循环 =====
encoder = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
decoder = Decoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
model   = Seq2Seq(encoder, decoder)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss(ignore_index=PAD)

for epoch in range(1, 11):
    model.train()
    total_loss = 0
    for _ in range(200):
        src, tgt = make_batch(BATCH_SIZE)
        # TODO: 前向传播、计算 loss、反向传播、更新参数
        # loss 计算：outputs[:,1:,:] 对比 tgt[:,1:]
        pass

    acc = accuracy(model)
    print(f'Epoch {epoch:2d} | loss={total_loss/200:.4f} | val_acc={acc:.1%}')
    if acc > 0.95:
        print('通关！')
        break

print('通关条件：val_acc > 95%')

## 练习 8.4：默写练习（不看资料）

**关闭 lesson.ipynb**，凭记忆完成以下默写。完成后再对照 lesson 检查。

In [ ]:
# 默写练习——用注释写出答案，不需要实际运行

# Q1: Seq2Seq 的核心思想是什么？（一句话）
# A1:

# Q2: Context Vector 是什么？它的 shape 是什么？
# A2: encoder 的输出 hidden (1, batch_size, hidden_size)

# Q3: Teacher Forcing 是什么？训练和推理有什么区别？
# A3: 在训练时，decoder的输入是标准答案；推理时，decoder的输入是上一步的预测值

# Q4: 为什么训练时用 CrossEntropyLoss 要从 outputs[:,1:,:] 开始而不是 outputs[:,0:,:]?
# A4: 因为outputs[:,0:,:] 包括了SOS，这个没必要计算损失，因为不属于预测值

# Q5: 曝光偏差（Exposure Bias）是什么问题？
# A5: 训练时使用标准答案做decoder输入，预测值和标准答案计算损失；推理时使用自己的上一次预测值做decoder输入，之前训练时没有经历过，第一次曝光在预测值输入下，产生的偏差

# Q6: 默写 Seq2Seq.forward() 的核心步骤（5步）：
# 步骤1:
# 步骤2:
# 步骤3:
# 步骤4:
# 步骤5:

print('默写完成后，和 lesson.ipynb 第 7 节对照检查')